In [ ]:
import json

# 너가 복사해온 JSON 데이터를 그대로 붙여넣기
notebook_json = {
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Import"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import random\n",
    "import glob\n",
    "import re\n",
    "from datetime import datetime, timedelta\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "\n",
    "from sklearn.preprocessing import LabelEncoder\n",
    "from sklearn.model_selection import train_test_split, TimeSeriesSplit\n",
    "from sklearn.metrics import mean_absolute_error, mean_squared_error\n",
    "\n",
    "import xgboost as xgb\n",
    "import optuna\n",
    "from tqdm import tqdm\n",
    "\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Fixed RandomSeed & Setting Parameters"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def set_seed(seed=42):\n",
    "    random.seed(seed)\n",
    "    np.random.seed(seed)\n",
    "    os.environ['PYTHONHASHSEED'] = str(seed)\n",
    "\n",
    "set_seed(42)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 하이퍼파라미터 설정\n",
    "LOOKBACK = 28  # 과거 28일 데이터 사용\n",
    "PREDICT = 7    # 미래 7일 예측\n",
    "N_TRIALS = 100 # OPTUNA 시행 횟수\n",
    "\n",
    "# XGBoost 관련 설정\n",
    "EARLY_STOPPING_ROUNDS = 50\n",
    "VERBOSE_EVAL = False"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Data Load and Preprocessing"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "'''\n",
    "Load dataset and preprocessing\n",
    "-> train | test | submission | prediction\n",
    "'''\n",
    "\n",
    "# train data\n",
    "train_data = pd.read_csv('./dataset/train/train.csv')\n",
    "# ['date'] -> datetime\n",
    "train_data['date'] = pd.to_datetime(train_data['date'], format='%Y-%m-%d')\n",
    "# ordinal date feature\n",
    "train_data['date_ordinal'] = train_data['date'].map(datetime.toordinal)\n",
    "# store_menu_id\n",
    "train_data['store_menu_id'] = train_data['store'] + \"_\" + train_data['menu']\n",
    "\n",
    "# test data\n",
    "for i in range(0, 10):\n",
    "    test = pd.read_csv(f\"./dataset/test/TEST_0{i}.csv\")\n",
    "    test['date'] = pd.to_datetime(test['date'], format='%Y-%m-%d')\n",
    "    test['date_ordinal'] = test['date'].map(datetime.toordinal)\n",
    "    test['store_menu_id'] = test['store'] + \"_\" + test['menu']\n",
    "    # test_data_{i} for all test datasets\n",
    "    globals()[f'test_data_{i}'] = test\n",
    "\n",
    "# submission format\n",
    "submission = pd.read_csv(\"./result/sample_submission_date.csv\")\n",
    "\n",
    "print(f\"Train data shape: {train_data.shape}\")\n",
    "print(f\"Train data date range: {train_data['date'].min()} ~ {train_data['date'].max()}\")\n",
    "print(f\"Unique store_menu combinations: {train_data['store_menu_id'].nunique()}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Feature Engineering"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def add_time_features(df):\n",
    "    \"\"\"시간 관련 피처 추가\"\"\"\n",
    "    df = df.copy()\n",
    "    \n",
    "    # 기본 시간 피처\n",
    "    df['year'] = df['date'].dt.year\n",
    "    df['month'] = df['date'].dt.month\n",
    "    df['day'] = df['date'].dt.day\n",
    "    df['dayofweek'] = df['date'].dt.dayofweek  # 0=월요일, 6=일요일\n",
    "    df['dayofyear'] = df['date'].dt.dayofyear\n",
    "    df['weekofyear'] = df['date'].dt.isocalendar().week\n",
    "    df['quarter'] = df['date'].dt.quarter\n",
    "    \n",
    "    # 주말/평일 구분\n",
    "    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)\n",
    "    df['is_monday'] = (df['dayofweek'] == 0).astype(int)\n",
    "    df['is_friday'] = (df['dayofweek'] == 4).astype(int)\n",
    "    \n",
    "    # 계절 정보\n",
    "    df['season'] = df['month'].map({12:0, 1:0, 2:0,  # 겨울\n",
    "                                   3:1, 4:1, 5:1,   # 봄\n",
    "                                   6:2, 7:2, 8:2,   # 여름\n",
    "                                   9:3, 10:3, 11:3}) # 가을\n",
    "    \n",
    "    # 월말/월초 구분\n",
    "    df['is_month_start'] = (df['day'] <= 5).astype(int)\n",
    "    df['is_month_end'] = (df['day'] >= 25).astype(int)\n",
    "    \n",
    "    # 순환 인코딩 (주기적 특성 보존)\n",
    "    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)\n",
    "    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)\n",
    "    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)\n",
    "    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)\n",
    "    df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)\n",
    "    df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)\n",
    "    \n",
    "    return df\n",
    "\n",
    "def add_lag_features(df, target_col='sales', lags=[1, 2, 3, 7, 14, 21, 28]):\n",
    "    \"\"\"지연 피처 (과거 값) 추가\"\"\"\n",
    "    df = df.copy()\n",
    "    df = df.sort_values(['store_menu_id', 'date'])\n",
    "    \n",
    "    for lag in lags:\n",
    "        df[f'{target_col}_lag_{lag}'] = df.groupby('store_menu_id')[target_col].shift(lag)\n",
    "    \n",
    "    return df\n",
    "\n",
    "def add_rolling_features(df, target_col='sales', windows=[3, 7, 14, 28]):\n",
    "    \"\"\"이동평균/이동통계 피처 추가\"\"\"\n",
    "    df = df.copy()\n",
    "    df = df.sort_values(['store_menu_id', 'date'])\n",
    "    \n",
    "    for window in windows:\n",
    "        # 이동평균\n",
    "        df[f'{target_col}_rolling_mean_{window}'] = (\n",
    "            df.groupby('store_menu_id')[target_col]\n",
    "            .rolling(window=window, min_periods=1)\n",
    "            .mean()\n",
    "            .reset_index(0, drop=True)\n",
    "        )\n",
    "        \n",
    "        # 이동표준편차\n",
    "        df[f'{target_col}_rolling_std_{window}'] = (\n",
    "            df.groupby('store_menu_id')[target_col]\n",
    "            .rolling(window=window, min_periods=1)\n",
    "            .std()\n",
    "            .reset_index(0, drop=True)\n",
    "        )\n",
    "        \n",
    "        # 이동 최대/최소\n",
    "        df[f'{target_col}_rolling_max_{window}'] = (\n",
    "            df.groupby('store_menu_id')[target_col]\n",
    "            .rolling(window=window, min_periods=1)\n",
    "            .max()\n",
    "            .reset_index(0, drop=True)\n",
    "        )\n",
    "        \n",
    "        df[f'{target_col}_rolling_min_{window}'] = (\n",
    "            df.groupby('store_menu_id')[target_col]\n",
    "            .rolling(window=window, min_periods=1)\n",
    "            .min()\n",
    "            .reset_index(0, drop=True)\n",
    "        )\n",
    "    \n",
    "    return df\n",
    "\n",
    "def add_statistical_features(df, target_col='sales'):\n",
    "    \"\"\"통계적 피처 추가\"\"\"\n",
    "    df = df.copy()\n",
    "    \n",
    "    # 전체 기간 통계\n",
    "    stats = df.groupby('store_menu_id')[target_col].agg([\n",
    "        'mean', 'std', 'min', 'max', 'median'\n",
    "    ]).add_prefix(f'{target_col}_overall_')\n",
    "    \n",
    "    df = df.merge(stats, on='store_menu_id', how='left')\n",
    "    \n",
    "    # 요일별 평균\n",
    "    dayofweek_stats = df.groupby(['store_menu_id', 'dayofweek'])[target_col].mean().reset_index()\n",
    "    dayofweek_stats = dayofweek_stats.pivot(index='store_menu_id', \n",
    "                                           columns='dayofweek', \n",
    "                                           values=target_col)\n",
    "    dayofweek_stats.columns = [f'{target_col}_dayofweek_mean_{int(col)}' for col in dayofweek_stats.columns]\n",
    "    \n",
    "    df = df.merge(dayofweek_stats, on='store_menu_id', how='left')\n",
    "    \n",
    "    # 월별 평균\n",
    "    month_stats = df.groupby(['store_menu_id', 'month'])[target_col].mean().reset_index()\n",
    "    month_stats = month_stats.pivot(index='store_menu_id', \n",
    "                                   columns='month', \n",
    "                                   values=target_col)\n",
    "    month_stats.columns = [f'{target_col}_month_mean_{int(col)}' for col in month_stats.columns]\n",
    "    \n",
    "    df = df.merge(month_stats, on='store_menu_id', how='left')\n",
    "    \n",
    "    return df"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def create_features(df, is_train=True):\n",
    "    \"\"\"전체 피처 생성 파이프라인\"\"\"\n",
    "    print(\"피처 엔지니어링 시작...\")\n",
    "    \n",
    "    # 시간 피처 추가\n",
    "    df = add_time_features(df)\n",
    "    print(\"시간 피처 추가 완료\")\n",
    "    \n",
    "    if is_train:\n",
    "        # 통계적 피처 추가 (훈련 데이터에서만)\n",
    "        df = add_statistical_features(df)\n",
    "        print(\"통계적 피처 추가 완료\")\n",
    "    \n",
    "    # 지연 피처 추가\n",
    "    df = add_lag_features(df)\n",
    "    print(\"지연 피처 추가 완료\")\n",
    "    \n",
    "    # 이동평균 피처 추가\n",
    "    df = add_rolling_features(df)\n",
    "    print(\"이동평균 피처 추가 완료\")\n",
    "    \n",
    "    # 카테고리컬 인코딩\n",
    "    le_store = LabelEncoder()\n",
    "    le_menu = LabelEncoder()\n",
    "    \n",
    "    if is_train:\n",
    "        df['store_encoded'] = le_store.fit_transform(df['store'])\n",
    "        df['menu_encoded'] = le_menu.fit_transform(df['menu'])\n",
    "        \n",
    "        # 인코더 저장\n",
    "        return df, le_store, le_menu\n",
    "    else:\n",
    "        return df\n",
    "\n",
    "# 훈련 데이터에 피처 추가\n",
    "train_featured, le_store, le_menu = create_features(train_data, is_train=True)\n",
    "print(f\"\\n훈련 데이터 피처 생성 완료: {train_featured.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# XGBoost Model Training with OPTUNA"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def prepare_xgboost_data(df, target_col='sales', exclude_cols=None):\n",
    "    \"\"\"XGBoost 학습용 데이터 준비\"\"\"\n",
    "    if exclude_cols is None:\n",
    "        exclude_cols = ['date', 'store', 'menu', 'store_menu_id']\n",
    "    \n",
    "    # 피처 선택\n",
    "    feature_cols = [col for col in df.columns if col not in exclude_cols + [target_col]]\n",
    "    \n",
    "    X = df[feature_cols]\n",
    "    y = df[target_col] if target_col in df.columns else None\n",
    "    \n",
    "    # 결측값 처리\n",
    "    X = X.fillna(0)\n",
    "    \n",
    "    return X, y, feature_cols\n",
    "\n",
    "def objective(trial):\n",
    "    \"\"\"OPTUNA 목적 함수\"\"\"\n",
    "    # 하이퍼파라미터 제안\n",
    "    params = {\n",
    "        'objective': 'reg:squarederror',\n",
    "        'eval_metric': 'rmse',\n",
    "        'booster': 'gbtree',\n",
    "        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),\n",
    "        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),\n",
    "        'subsample': trial.suggest_float('subsample', 0.5, 1.0),\n",
    "        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),\n",
    "        'max_depth': trial.suggest_int('max_depth', 3, 10),\n",
    "        'min_child_weight': trial.suggest_int('min_child_weight', 1, 100),\n",
    "        'eta': trial.suggest_float('eta', 0.01, 0.3, log=True),\n",
    "        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),\n",
    "        'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),\n",
    "        'seed': 42,\n",
    "        'verbosity': 0\n",
    "    }\n",
    "    \n",
    "    # 샘플 데이터로 빠른 검증\n",
    "    sample_stores = train_featured['store_menu_id'].unique()[:50]  # 50개 store_menu만 사용\n",
    "    sample_data = train_featured[train_featured['store_menu_id'].isin(sample_stores)].copy()\n",
    "    \n",
    "    # 시계열 분할 검증\n",
    "    tscv = TimeSeriesSplit(n_splits=3)\n",
    "    cv_scores = []\n",
    "    \n",
    "    for store_menu, group in sample_data.groupby('store_menu_id'):\n",
    "        if len(group) < 50:  # 최소 데이터 요구량\n",
    "            continue\n",
    "            \n",
    "        group = group.sort_values('date')\n",
    "        X, y, _ = prepare_xgboost_data(group)\n",
    "        \n",
    "        if X.shape[0] < 30:  # 최소 샘플 수 확인\n",
    "            continue\n",
    "        \n",
    "        # 시계열 교차 검증\n",
    "        for train_idx, val_idx in tscv.split(X):\n",
    "            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]\n",
    "            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]\n",
    "            \n",
    "            # XGBoost 데이터셋 생성\n",
    "            dtrain = xgb.DMatrix(X_train_fold, label=y_train_fold)\n",
    "            dval = xgb.DMatrix(X_val_fold, label=y_val_fold)\n",
    "            \n",
    "            # 학습\n",
    "            model = xgb.train(\n",
    "                params,\n",
    "                dtrain,\n",
    "                num_boost_round=1000,\n",
    "                evals=[(dval, 'eval')],\n",
    "                early_stopping_rounds=50,\n",
    "                verbose_eval=False\n",
    "            )\n",
    "            \n",
    "            # 예측 및 평가\n",
    "            y_pred = model.predict(dval)\n",
    "            rmse = np.sqrt(mean_squared_error(y_val_fold, y_pred))\n",
    "            cv_scores.append(rmse)\n",
    "            \n",
    "            # 너무 많은 검증을 방지\n",
    "            if len(cv_scores) >= 10:\n",
    "                break\n",
    "        \n",
    "        if len(cv_scores) >= 10:\n",
    "            break\n",
    "    \n",
    "    if len(cv_scores) == 0:\n",
    "        return float('inf')\n",
    "    \n",
    "    return np.mean(cv_scores)\n",
    "\n",
    "# OPTUNA 하이퍼파라미터 튜닝\n",
    "print(\"하이퍼파라미터 튜닝 시작...\")\n",
    "study = optuna.create_study(direction='minimize')\n",
    "study.optimize(objective, n_trials=N_TRIALS)\n",
    "\n",
    "print(f\"\\n=== 최적 하이퍼파라미터 ===\")\n",
    "print(f\"Best value: {study.best_value:.4f}\")\n",
    "print(f\"Best params: {study.best_params}\")\n",
    "\n",
    "best_params = study.best_params\n",
    "best_params.update({\n",
    "    'objective': 'reg:squarederror',\n",
    "    'eval_metric': 'rmse',\n",
    "    'seed': 42,\n",
    "    'verbosity': 0\n",
    "})"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Train XGBoost Models"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def train_xgboost_models(train_df, best_params):\n",
    "    \"\"\"각 store_menu별로 XGBoost 모델 훈련\"\"\"\n",
    "    trained_models = {}\n",
    "    \n",
    "    print(f\"최적 파라미터로 전체 모델 훈련 시작...\")\n",
    "    \n",
    "    for store_menu, group in tqdm(train_df.groupby('store_menu_id'), desc='Training XGBoost'):\n",
    "        if len(group) < LOOKBACK + PREDICT:\n",
    "            continue\n",
    "            \n",
    "        group = group.sort_values('date')\n",
    "        \n",
    "        # 다중 스텝 예측을 위한 데이터 준비\n",
    "        models_for_days = {}\n",
    "        \n",
    "        for day_ahead in range(1, PREDICT + 1):\n",
    "            # day_ahead일 후의 값을 예측하는 모델\n",
    "            X, y, feature_cols = prepare_xgboost_data(group)\n",
    "            \n",
    "            # day_ahead만큼 시프트한 타겟 생성\n",
    "            y_shifted = group['sales'].shift(-day_ahead)\n",
    "            \n",
    "            # 유효한 데이터만 선택\n",
    "            valid_idx = ~y_shifted.isna()\n",
    "            X_valid = X[valid_idx]\n",
    "            y_valid = y_shifted[valid_idx]\n",
    "            \n",
    "            if len(X_valid) < 10:  # 최소 데이터 요구량\n",
    "                continue\n",
    "            \n",
    "            # 훈련/검증 분할 (시계열 고려)\n",
    "            split_idx = int(len(X_valid) * 0.8)\n",
    "            X_train = X_valid.iloc[:split_idx]\n",
    "            X_val = X_valid.iloc[split_idx:]\n",
    "            y_train = y_valid.iloc[:split_idx]\n",
    "            y_val = y_valid.iloc[split_idx:]\n",
    "            \n",
    "            if len(X_val) == 0:\n",
    "                X_train = X_valid\n",
    "                y_train = y_valid\n",
    "                X_val = X_valid.iloc[-5:]  # 마지막 5개를 검증용으로\n",
    "                y_val = y_valid.iloc[-5:]\n",
    "            \n",
    "            # XGBoost 데이터셋 생성\n",
    "            dtrain = xgb.DMatrix(X_train, label=y_train)\n",
    "            dval = xgb.DMatrix(X_val, label=y_val)\n",
    "            \n",
    "            # 모델 훈련\n",
    "            model = xgb.train(\n",
    "                best_params,\n",
    "                dtrain,\n",
    "                num_boost_round=1000,\n",
    "                evals=[(dval, 'eval')],\n",
    "                early_stopping_rounds=EARLY_STOPPING_ROUNDS,\n",
    "                verbose_eval=VERBOSE_EVAL\n",
    "            )\n",
    "            \n",
    "            models_for_days[day_ahead] = model\n",
    "        \n",
    "        if models_for_days:\n",
    "            trained_models[store_menu] = {\n",
    "                'models': models_for_days,\n",
    "                'feature_cols': feature_cols,\n",
    "                'last_data': group.iloc[-LOOKBACK:]  # 마지막 28일 데이터\n",
    "            }\n",
    "    \n",
    "    return trained_models\n",
    "\n",
    "# 전체 모델 훈련\n",
    "trained_models = train_xgboost_models(train_featured, best_params)\n",
    "print(f\"\\n훈련 완료: {len(trained_models)}개 store_menu에 대한 모델\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Prediction"
   ]
  },
  
    }

# 저장 경로
# output_path = "gru_optuna_pipeline.ipynb"

# 파일로 저장
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(notebook_json, f, indent=1, ensure_ascii=False)

print(f"📁 저장 완료: {output_path}")
